<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/claude/adoring-thompson-Lv2MQ/notebooks/kaggle_environments_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforce Tactics — Kaggle Environments Smoke Test

End-to-end validation that the **Reinforce Tactics** Kaggle Environments adapter is wired up correctly.

This notebook:
1. Installs `kaggle-environments` and clones this branch.
2. Stages the adapter into `kaggle_environments/envs/reinforce_tactics/` — the same path `kaggle/kaggle-environments` PR #673 deploys to.
3. Runs a series of `make()` / `reset()` / `run()` smoke tests.
4. Executes both pytest suites (the 21 functional tests and the 90 comprehensive internal tests).

If every cell completes without errors, you have a working setup that mirrors what the upstream PR will produce.

## 1. Setup

Install `kaggle-environments`, clone this branch, then copy the adapter into the `kaggle_environments/envs/` directory before any first import. The import-time directory scan is how `make("reinforce_tactics")` discovers the environment.

In [ ]:
import importlib.util
import shutil
import subprocess

# Install kaggle-environments first — but DO NOT import it yet.
subprocess.run(["pip", "install", "-q", "kaggle-environments", "pytest"], check=True)

# Find the install path without importing the module.
ke_dir = importlib.util.find_spec("kaggle_environments").submodule_search_locations[0]
print("kaggle_environments install path:", ke_dir)

In [ ]:
# Clone this branch.
REPO_DIR = "/tmp/reinforce-tactics"
BRANCH = "claude/adoring-thompson-Lv2MQ"

subprocess.run(["rm", "-rf", REPO_DIR], check=False)
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "-b",
        BRANCH,
        "https://github.com/kuds/reinforce-tactics.git",
        REPO_DIR,
    ],
    check=True,
)

# Stage the adapter into kaggle_environments/envs/reinforce_tactics/.
target = f"{ke_dir}/envs/reinforce_tactics"
shutil.rmtree(target, ignore_errors=True)
shutil.copytree(f"{REPO_DIR}/reinforcetactics/kaggle", target)
print("Adapter staged at:", target)

## 2. Smoke tests via the public API

Now that the adapter is on disk, importing `kaggle_environments` will register it and `make("reinforce_tactics")` will return a working `Environment`.

In [ ]:
from kaggle_environments import evaluate, make

env = make(
    "reinforce_tactics",
    configuration={"mapName": "beginner", "episodeSteps": 30, "mapSeed": 42},
    debug=False,
)
state = env.reset()

print("Number of agent slots :", len(state))
print("Player 0 status      :", state[0]["status"])
print("Player 1 status      :", state[1]["status"])
print("Starting gold        :", state[0]["observation"]["gold"])
print("Map name (config)    :", env.configuration.mapName)
print("Board size           :", len(state[0]["observation"]["board"]), "x", len(state[0]["observation"]["board"][0]))

structures = state[0]["observation"]["structures"]
print("Structure count      :", len(structures))
print("Structure types      :", sorted({s["type"] for s in structures}))

In [ ]:
# ASCII render of the initial board
print(env.render(mode="ansi"))

In [ ]:
# Run a full episode: random agent vs the strategic simple_bot.
result = env.run(["random", "simple_bot"])
final = result[-1]

print(f"Turns simulated      : {len(result) - 1}")
print(f"Final status (P0/P1) : {final[0].status} / {final[1].status}")
print(f"Final reward (P0/P1) : {final[0].reward} / {final[1].reward}")

if final[0].reward == final[1].reward == 0:
    print("Outcome              : Draw or step-cap reached")
elif final[0].reward > final[1].reward:
    print("Outcome              : random agent won")
else:
    print("Outcome              : simple_bot won")

## 3. Configuration variations

Spot-check that the configurable knobs all work: the three built-in maps, random generation, fog of war, custom starting gold, and the `evaluate()` helper.

In [ ]:
# Every built-in map should reset cleanly to a >=20x20 board (small maps are
# padded). All 19 two-player maps are vendored from the repo's maps/1v1/.
from kaggle_environments.envs.reinforce_tactics.reinforce_tactics import BUILTIN_MAPS

for name in sorted(BUILTIN_MAPS):
    e = make("reinforce_tactics", configuration={"mapName": name})
    s = e.reset()
    rows = len(s[0]["observation"]["board"])
    cols = len(s[0]["observation"]["board"][0])
    assert rows >= 20 and cols >= 20, (name, rows, cols)
    print(f"  {name:<18} -> {rows}x{cols} board, status={s[0]['status']}")
print(f"  all {len(BUILTIN_MAPS)} built-in maps reset OK")

In [ ]:
# Random map (mapName='') and fog of war.
rand_env = make("reinforce_tactics", configuration={"mapName": "", "mapSeed": 7})
rand_state = rand_env.reset()
ocean_count = sum(row.count("o") for row in rand_state[0]["observation"]["board"])
print(f"Random map ocean tiles: {ocean_count}")

fog_env = make("reinforce_tactics", configuration={"fogOfWar": True, "mapSeed": 42})
fog_state = fog_env.reset()
print(f"Fog of war units list type: {type(fog_state[0]['observation']['units']).__name__}")

rich_env = make("reinforce_tactics", configuration={"startingGold": 1000, "mapSeed": 42})
rich_state = rich_env.reset()
print(f"Custom starting gold      : {rich_state[0]['observation']['gold']}")

In [ ]:
# evaluate(): play several episodes and collect rewards.
rewards = evaluate(
    "reinforce_tactics",
    ["random", "random"],
    num_episodes=3,
    configuration={"mapSeed": 42, "episodeSteps": 20},
)
print("Per-episode rewards (each row sums to 0):")
for i, r in enumerate(rewards):
    print(f"  Episode {i + 1}: P0={r[0]}, P1={r[1]}")

## 4. Engine-direct sanity checks

The kaggle interpreter doesn't exercise every code path in the vendored engine. The cells below hit the engine directly to confirm the fixes for the issues called out in code review:

* `get_legal_actions()` returns a dict (used to `AttributeError` on `_cache_valid`).
* Legal moves do not include destinations occupied by the moving player's own units.
* `end_turn()` honours `max_turns` and ends in a draw.
* `to_dict()` / `from_dict()` round-trip `original_x`/`original_y`, `action_history`, and `max_turns`.
* `resign()` strips the resigning player's units.
* `GameState` rejects `num_players != 2` (kaggle env is 2-player by spec).

In [ ]:
import sys

sys.path.insert(0, REPO_DIR)
import numpy as np
import pandas as pd

from reinforcetactics.kaggle.reinforce_tactics_engine import GameState


def small_map():
    m = np.array([["p"] * 10 for _ in range(10)], dtype=object)
    m[0, 0] = "h_1"
    m[0, 1] = "b_1"
    m[9, 9] = "h_2"
    m[9, 8] = "b_2"
    return pd.DataFrame(m)


# Legal actions: returns dict, includes create_unit at the building
g = GameState(small_map(), num_players=2)
actions = g.get_legal_actions()
assert isinstance(actions, dict)
assert any((a["x"], a["y"]) == (1, 0) for a in actions["create_unit"])
print("legal_actions OK -> create_unit at building exposed")

# Move filter: own-unit-occupied tile must NOT appear as a legal destination
g2 = GameState(small_map(), num_players=2)
g2.player_gold = {1: 5000, 2: 5000}
ally_a = g2.create_unit("W", 4, 4, player=1)
g2.create_unit("W", 5, 4, player=1)
ally_a.can_move = True
moves = [m for m in g2.get_legal_actions(player=1)["move"] if m["unit"] is ally_a]
destinations = {(m["to_x"], m["to_y"]) for m in moves}
assert (5, 4) not in destinations
print("move_filter OK -> ally-occupied tile excluded")

In [ ]:
# max_turns triggers a draw with winner=None
g3 = GameState(small_map(), num_players=2, max_turns=3)
for _ in range(20):
    if g3.game_over:
        break
    g3.end_turn()
assert g3.game_over is True and g3.winner is None
print(f"max_turns OK -> game_over={g3.game_over}, winner={g3.winner} (draw)")

# resign() strips units and sets winner
g4 = GameState(small_map(), num_players=2)
g4.player_gold = {1: 5000, 2: 5000}
g4.create_unit("W", 3, 3, player=1)
g4.create_unit("M", 6, 6, player=2)
g4.resign(player=1)
assert g4.game_over and g4.winner == 2 and all(u.player != 1 for u in g4.units)
print(f"resign OK -> winner={g4.winner}, surviving units={[u.player for u in g4.units]}")

# 2-player assertion
try:
    GameState(small_map(), num_players=3)
    raise AssertionError("expected ValueError")
except ValueError as exc:
    print(f"assert_2_player OK -> {exc}")

In [ ]:
# to_dict / from_dict round-trip: max_turns, action_history, original_x/y, and
# the engine_overrides overlay (persisted so a reloaded game keeps its balance).
g5 = GameState(
    small_map(), num_players=2, max_turns=42, engine_overrides={"damage_model": "hp_scaled", "max_units_per_player": 7}
)
g5.player_gold = {1: 5000, 2: 5000}
g5.create_unit("W", 3, 3, player=1)
warrior = next(u for u in g5.units if u.type == "W")
warrior.x, warrior.y = 4, 3
warrior.has_moved = True
g5.record_action("custom", note="colab")

payload = g5.to_dict()
assert "action_history" in payload and "max_turns" in payload
restored = GameState.from_dict(payload, small_map())
assert restored.max_turns == 42
assert restored.engine_overrides == {"damage_model": "hp_scaled", "max_units_per_player": 7}
assert restored.damage_model == "hp_scaled" and restored.max_units_per_player == 7
assert any(a.get("type") == "custom" for a in restored.action_history)
warrior_r = next(u for u in restored.units if u.type == "W")
assert (warrior_r.original_x, warrior_r.original_y) != (warrior_r.x, warrior_r.y)
print("save_load_round_trip OK -> max_turns + action_history + original_x/y + engine_overrides preserved")

In [ ]:
# Competition balance: the adapter applies v52a's engine_overrides to every
# game it builds (Warrior cost 300, hp_scaled combat); see ENGINE_OVERRIDES.
import types

from reinforcetactics.kaggle.reinforce_tactics import ENGINE_OVERRIDES, _init_game

cfg = types.SimpleNamespace(
    episodeSteps=200,
    actTimeout=5,
    runTimeout=1200,
    mapName="",
    mapWidth=20,
    mapHeight=20,
    mapSeed=42,
    enabledUnits="W,M,C,A,K,R,S,B",
    fogOfWar=False,
    startingGold=250,
)
gv = _init_game(cfg)
assert gv.unit_data["W"]["cost"] == 300, gv.unit_data["W"]["cost"]
assert gv.damage_model == "hp_scaled", gv.damage_model
assert gv.unit_data["A"]["cost"] == 250  # non-overridden units keep engine defaults
print(
    "v52a_balance OK -> Warrior cost",
    gv.unit_data["W"]["cost"],
    "| damage_model",
    gv.damage_model,
    "| overrides",
    ENGINE_OVERRIDES,
)

## 5. Run the unit-test suites

Three test suites ship in the staging repo:

* **`tests/test_kaggle_env_make.py`** — 21 functional tests that exercise the env through `kaggle_environments.make()`, mirroring the test file that will live at `tests/envs/reinforce_tactics/test_reinforce_tactics.py` once PR #673 merges upstream.
* **`tests/test_kaggle_env.py`** — 91 unit tests against the adapter internals (interpreter, serialisation, action execution, win/draw conditions, built-in agents). Staging-only.
* **`tests/test_kaggle_engine.py`** — 13 engine-direct regression tests guarding against the code-review findings (legal actions, max_turns draw, save/load round-trip, fog-of-war memory clear, resign behaviour). Staging-only.

All three run from the cloned repo. We pass `--override-ini=addopts=` to neutralise this project's `pytest-cov` config (which expects the full ML stack to be importable).

In [ ]:
import os

test_files = [
    ("functional (make)", "tests/test_kaggle_env_make.py", "q"),
    ("comprehensive (interp)", "tests/test_kaggle_env.py", "q"),
    ("engine regression", "tests/test_kaggle_engine.py", "v"),
]

for label, path, verbosity in test_files:
    print(f"\n=== {label}: {path} ===")
    result = subprocess.run(
        ["pytest", path, f"-{verbosity}", "--override-ini=addopts=", "--no-header", "-p", "no:cacheprovider"],
        cwd=REPO_DIR,
        env={**os.environ, "PYTHONPATH": REPO_DIR},
        capture_output=True,
        text=True,
    )
    # Show full output for engine regression (it's small) and tail for the others.
    out = result.stdout if verbosity == "v" else result.stdout[-1500:]
    print(out)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise SystemExit(f"{label} failed with exit code {result.returncode}")

print("\nAll suites: PASS")

## Done

If every cell ran without errors, the staged adapter behaves correctly through `kaggle_environments.make()`, the engine-direct sanity checks pass, and all three test suites are green. This is the same shape PR #673 produces once merged.

**Next steps:**
* Browse `kaggle_environments/envs/reinforce_tactics/README.md` (now staged in your runtime) for the action / observation reference.
* Submit your own agent: write a function `agent(observation, configuration) -> list[dict]` and pass it to `env.run([my_agent, 'random'])`.
* See `notebooks/bot_tournament.ipynb` for higher-level evaluation tooling.